# Blue-and-White Ceramics Retrieval in Dutch Golden Age Paintings

This notebook runs the full retrieval experiment: zero-shot CLIP, RAG-augmented CLIP,
KB ablation, and metadata keyword baselines. All evaluated against manually annotated
ground truth.

**Prerequisites:** Pre-computed data files in `data/`:
- `painting_embeddings.npz` — CLIP embeddings for 150 paintings
- `paintings_metadata.json` — metadata (title, maker, id, img_url) for 150 paintings
- `kb_embeddings.npz` — CLIP embeddings for KB images (paintings + ceramic photos)
- `annotations.csv` — text annotations for KB items
- `ground_truth.json` — manually annotated painting IDs containing blue-and-white ceramics

See `tools/image2embedding.ipynb` for how embeddings were generated,
and `tools/fetch_paintings.ipynb` for how painting metadata was collected.

## 0. Setup

In [1]:
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

import json
import csv
import numpy as np
import clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
print(f"CLIP loaded on {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.2 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-8itk2xts
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-8itk2xts
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=95992183d8a5685db874f9ef45ff2f8ad02ae49aa28f3a4a828cfa2b6c453126
  Stored in directory: /tmp/pip-ephem-wheel-cache-3aqhp8t6/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 104MiB/s]


CLIP loaded on cuda


## 1. Load Pre-computed Data

In [5]:
# Painting embeddings and metadata
painting_emb_data = np.load("data/painting_embeddings.npz")
painting_embeddings = dict(zip(
    painting_emb_data["ids"],
    painting_emb_data["embeddings"].astype(np.float32)
))

with open("data/paintings_metadata.json") as f:
    paintings_dict = json.load(f)  # {url_id: {title, maker, ...}}
paintings = [{"id": pid, **meta} for pid, meta in paintings_dict.items()]

# KB embeddings (key-per-embedding format: c1, c2, m1, m2, ...)
kb_data = np.load("data/kb_embeddings.npz")
kb_embeddings = {key: kb_data[key] for key in kb_data.files}

# KB annotations
annotation_lookup = {}
with open("data/annotations.csv") as f:
    for row in csv.DictReader(f):
        item_id = row["id"]
        anns = [row[col] for col in ["annotation_1", "annotation_2"] if row.get(col)]
        if anns:
            annotation_lookup[item_id] = anns

# Ground truth
with open("data/ground_truth.json") as f:
    GROUND_TRUTH = set(json.load(f))

print(f"Paintings: {len(paintings)}")
print(f"Painting embeddings: {len(painting_embeddings)}")
print(f"KB items: {len(kb_embeddings)}")
print(f"Ground truth: {len(GROUND_TRUTH)} paintings with blue-and-white ceramics")

Paintings: 150
Painting embeddings: 150
KB items: 30
Ground truth: 28 paintings with blue-and-white ceramics


## 2. Define Prompts and Helpers

In [9]:
PROMPTS = [
    "blue and white porcelain bowl",
    "blue and white porcelain plate",
    "blue and white ceramic dish on a table",
    "a bowl with blue floral decoration",
    "a vase with blue floral decoration",
    "a still life painting with blue and white pottery",
    "a painting of a table with porcelain dishes",
    "white dish with blue painted pattern",
    "round plate with blue decorative motifs",
]


def encode_prompts(prompts):
    """Encode text prompts into a single averaged CLIP query vector."""
    tokens = clip.tokenize(prompts, truncate=True).to(device)
    with torch.no_grad():
        feats = model.encode_text(tokens)
        feats /= feats.norm(dim=-1, keepdim=True)
    query = feats.mean(dim=0, keepdim=True)
    query /= query.norm(dim=-1, keepdim=True)
    return query.float()


def score_paintings(query_vec, paintings, painting_embeddings):
    """Score all paintings against a query using pre-computed embeddings."""
    results = []
    for p in paintings:
        pid = p["id"]
        if pid not in painting_embeddings:
            continue
        img_feat = torch.tensor(painting_embeddings[pid], dtype=torch.float32, device=device).unsqueeze(0)
        img_feat /= img_feat.norm(dim=-1, keepdim=True)
        score = (img_feat @ query_vec.T).item()
        results.append({
            "title": p.get("title", "Untitled"),
            "maker": p.get("maker", "Unknown"),
            "score": score,
            "page_url": pid,
        })
    return sorted(results, key=lambda x: x["score"], reverse=True)


def evaluate(ranked_results, ground_truth, k_values=[5, 10, 20]):
    """Compute precision@k, recall@k, and MRR."""
    metrics = {}
    ids = [r["page_url"] for r in ranked_results]

    # MRR
    mrr = 0.0
    for i, pid in enumerate(ids):
        if pid in ground_truth:
            mrr = 1.0 / (i + 1)
            break
    metrics["MRR"] = mrr

    for k in k_values:
        top_k_ids = set(ids[:k])
        hits = top_k_ids & ground_truth
        metrics[f"P@{k}"] = len(hits) / k
        metrics[f"R@{k}"] = len(hits) / len(ground_truth)
    return metrics

## 3. Zero-shot CLIP Retrieval

In [23]:
base_query = encode_prompts(PROMPTS)
results_base = score_paintings(base_query, paintings, painting_embeddings)

print("Top 10 — Zero-shot:")
for r in results_base[:10]:
    marker = "*" if r["page_url"] in GROUND_TRUTH else " "
    print(f"  {marker} {r['score']:.3f} | {r['title']}")
    print(f"           {r['page_url']}")

Top 10 — Zero-shot:
  * 0.314 | Stilleven
           https://id.rijksmuseum.nl/20015914
  * 0.305 | Still Life
           https://id.rijksmuseum.nl/20029259
    0.303 | Stilleven met bloemen en vruchten
           https://id.rijksmuseum.nl/20028752
    0.298 | Stilleven met bloemen, vruchten en gevogelte
           https://id.rijksmuseum.nl/200108670
    0.297 | Still Life with Flowers
           https://id.rijksmuseum.nl/200110737
  * 0.292 | Stilleven met kan van steengoed
           https://id.rijksmuseum.nl/20028547
    0.289 | Still Life with Flowers
           https://id.rijksmuseum.nl/200109818
  * 0.288 | Stilleven met vruchten en vaatwerk op een Smyrna kleed
           https://id.rijksmuseum.nl/20027878
  * 0.287 | Stilleven met vruchten, oesters en een porseleinen kom
           https://id.rijksmuseum.nl/20027535
    0.285 | Still Life with Flowers and Fruit
           https://id.rijksmuseum.nl/20027188


## 4. RAG-augmented CLIP Retrieval

In [22]:
def build_rag_query(kb_ids, base_query, top_k=5):
    """Retrieve top-k KB items, augment prompts with their annotations."""
    subset = {k: kb_embeddings[k] for k in kb_ids if k in kb_embeddings}
    if not subset:
        return base_query

    ids_list = list(subset.keys())
    matrix = np.stack([subset[k] for k in ids_list])
    scores = matrix @ base_query.cpu().numpy().squeeze()
    top_indices = np.argsort(scores)[::-1][:min(top_k, len(ids_list))]

    retrieved = []
    for idx in top_indices:
        anns = annotation_lookup.get(ids_list[idx], [])
        retrieved.extend(anns)

    augmented_prompts = PROMPTS + retrieved
    print(f"  Augmented: {len(PROMPTS)} base + {len(retrieved)} retrieved = {len(augmented_prompts)} prompts")
    return encode_prompts(augmented_prompts)


# Full KB RAG
print("Building RAG query (full KB):")
rag_query = build_rag_query(list(kb_embeddings.keys()), base_query)
results_rag = score_paintings(rag_query, paintings, painting_embeddings)

shift = (base_query.cpu() @ rag_query.cpu().T).item()
print(f"Cosine similarity between base and RAG query: {shift:.4f}")

print("\nTop 10 — RAG-augmented:")
for r in results_rag[:10]:
    marker = "*" if r["page_url"] in GROUND_TRUTH else " "
    print(f"  {marker} {r['score']:.3f} | {r['title']}")
    print(f"           {r['page_url']}")

Building RAG query (full KB):
  Augmented: 9 base + 10 retrieved = 19 prompts
Cosine similarity between base and RAG query: 0.9235

Top 10 — RAG-augmented:
    0.352 | Stilleven met bloemen en vruchten
           https://id.rijksmuseum.nl/20028752
    0.328 | Stilleven met bloemen, vruchten en gevogelte
           https://id.rijksmuseum.nl/200108670
    0.323 | Still Life with Flowers
           https://id.rijksmuseum.nl/200109818
  * 0.312 | Stilleven
           https://id.rijksmuseum.nl/20015914
    0.311 | The Overturned Bouquet
           https://id.rijksmuseum.nl/20027069
    0.308 | Still Life with Flowers
           https://id.rijksmuseum.nl/200110737
    0.308 | Still Life with Flowers and Fruit
           https://id.rijksmuseum.nl/20027842
    0.307 | Still Life with Flowers and Fruit
           https://id.rijksmuseum.nl/20027188
    0.307 | Stilleven met vruchten en een puttertje
           https://id.rijksmuseum.nl/200108474
    0.307 | Stilleven met aardbeien in een witte s

## 5. Ablation: KB Subsets

In [12]:
ceramic_ids = [k for k in kb_embeddings if k.startswith("c")]
painting_ids_kb = [k for k in kb_embeddings if not k.startswith("c")]
print(f"KB breakdown: {len(painting_ids_kb)} paintings, {len(ceramic_ids)} ceramic photos")

print("\nBuilding paintings-only RAG query:")
rag_paintings = build_rag_query(painting_ids_kb, base_query)
results_paintings_only = score_paintings(rag_paintings, paintings, painting_embeddings)

print("Building ceramics-only RAG query:")
rag_ceramics = build_rag_query(ceramic_ids, base_query)
results_ceramics_only = score_paintings(rag_ceramics, paintings, painting_embeddings)

KB breakdown: 20 paintings, 10 ceramic photos

Building paintings-only RAG query:
  Augmented: 9 base + 10 retrieved = 19 prompts
Building ceramics-only RAG query:
  Augmented: 9 base + 10 retrieved = 19 prompts


In [21]:
print("\nTop 10 — RAG (paintings-only KB):")
for r in results_paintings_only[:10]:
    marker = "*" if r["page_url"] in GROUND_TRUTH else " "
    print(f"  {marker} {r['score']:.3f} | {r['title']}")
    print(f"           {r['page_url']}")


Top 10 — RAG (paintings-only KB):
  * 0.421 | Stilleven
           https://id.rijksmuseum.nl/20015914
  * 0.404 | Stilleven met vruchten en vaatwerk op een Smyrna kleed
           https://id.rijksmuseum.nl/20027878
  * 0.403 | Still Life
           https://id.rijksmuseum.nl/20029259
  * 0.389 | Stilleven met vruchten, oesters en een porseleinen kom
           https://id.rijksmuseum.nl/20027535
  * 0.389 | Stilleven met kan van steengoed
           https://id.rijksmuseum.nl/20028547
    0.389 | Stilleven met vruchten en een puttertje
           https://id.rijksmuseum.nl/200108474
    0.384 | Stilleven met een verguld zilveren bekerschroef
           https://id.rijksmuseum.nl/20065885
  * 0.384 | Stilleven met vruchten
           https://id.rijksmuseum.nl/200109351
    0.380 | Stilleven met Nautilusbeker
           https://id.rijksmuseum.nl/200107972
    0.380 | Stilleven met bloemen, vruchten en gevogelte
           https://id.rijksmuseum.nl/200108670


## 6. Metadata Keyword Baseline

In [13]:
KEYWORD_SETS = {
    "general": [
        "porselein", "porcelein", "porcelain",
        "keramiek", "ceramic",
        "aardewerk", "earthenware",
    ],
    "basic": [
        "Chinees porselein", "Chinese porcelain",
        "blauw en wit", "blue and white",
    ],
    "expert": [
        "Chinees porselein", "Chinese porcelain",
        "blauw en wit", "blue and white",
        "kraak", "wan-li", "wanli", "Delfts blauw",
    ],
}


def keyword_score(title, maker, all_text, phrases):
    text = f"{title} {maker} {all_text}".lower()
    return sum(1 for phrase in phrases if phrase.lower() in text)


# all_text is already included in paintings_metadata.json
# Score with each keyword set
results_meta = {}
for name, phrases in KEYWORD_SETS.items():
    scored = []
    for p in paintings:
        s = keyword_score(p.get("title", ""), p.get("maker", ""), p.get("all_text", ""), phrases)
        scored.append({"title": p.get("title"), "page_url": p.get("id"), "score": s})
    scored.sort(key=lambda x: x["score"], reverse=True)
    results_meta[name] = [r for r in scored if r["score"] > 0]
    print(f"Meta-{name}: {len(results_meta[name])} results with hits")

Meta-general: 13 results with hits
Meta-basic: 1 results with hits
Meta-expert: 3 results with hits


## 7. Evaluation

In [14]:
# Collect all methods
all_methods = {
    "Zero-shot": results_base,
    "RAG (full KB)": results_rag,
    "RAG (paintings)": results_paintings_only,
    "RAG (ceramics)": results_ceramics_only,
    "Meta-general": results_meta["general"],
    "Meta-basic": results_meta["basic"],
    "Meta-expert": results_meta["expert"],
}

# Table: IR metrics
print(f"{'Method':<20} {'MRR':>6} {'P@5':>6} {'P@10':>6} {'P@20':>6} {'R@5':>6} {'R@10':>6} {'R@20':>6}")
print("-" * 78)
for name, results in all_methods.items():
    m = evaluate(results, GROUND_TRUTH)
    print(f"{name:<20} {m['MRR']:>6.3f} {m['P@5']:>6.3f} {m['P@10']:>6.3f} {m['P@20']:>6.3f} "
          f"{m['R@5']:>6.3f} {m['R@10']:>6.3f} {m['R@20']:>6.3f}")

Method                  MRR    P@5   P@10   P@20    R@5   R@10   R@20
------------------------------------------------------------------------------
Zero-shot             1.000  0.400  0.500  0.300  0.071  0.179  0.214
RAG (full KB)         0.250  0.200  0.100  0.300  0.036  0.036  0.214
RAG (paintings)       1.000  1.000  0.600  0.600  0.179  0.214  0.429
RAG (ceramics)        0.250  0.200  0.100  0.300  0.036  0.036  0.214
Meta-general          1.000  0.600  0.600  0.350  0.107  0.214  0.250
Meta-basic            1.000  0.200  0.100  0.050  0.036  0.036  0.036
Meta-expert           1.000  0.600  0.300  0.150  0.107  0.107  0.107


In [17]:
# Table: correct results at each cutoff (head-to-head)
def correct_at_k(results, k):
    return sum(1 for r in results[:k] if r['page_url'] in GROUND_TRUTH)

methods_subset = [
    ('Meta-general', results_meta['general']),
    ('Meta-expert', results_meta['expert']),
    ('Zero-shot', results_base),
    ('RAG-painting-KB)', results_paintings_only),
]

print(f"{'Cutoff':<12}", '  '.join(f'{n:>14}' for n, _ in methods_subset))
print('-' * 72)
for k in [5, 10, 20]:
    vals = []
    for name, results in methods_subset:
        max_n = len(results)
        c = correct_at_k(results, min(k, max_n))
        label = f'{c}/{max_n} (max)' if k > max_n else f'{c}/{k}'
        vals.append(f'{label:>14}')
    print(f'Top {k:<8}', '  '.join(vals))

Cutoff         Meta-general     Meta-expert       Zero-shot  RAG-painting-KB)
------------------------------------------------------------------------
Top 5                   3/5       3/3 (max)             2/5             5/5
Top 10                 6/10       3/3 (max)            5/10            6/10
Top 20           7/13 (max)       3/3 (max)            6/20           12/20


In [19]:
# Precision & Recall @ 20 comparison
print(f"{'Method':<20} {'Returned':>10} {'Correct':>10} {'Precision':>10} {'Recall':>10}")
print("-" * 62)

for name, results in [
    ("Zero-shot @20", results_base[:20]),
    ("RAG-painting-KB @20", results_paintings_only[:20]),
    ("Meta-general", results_meta["general"]),
    ("Meta-basic", results_meta["basic"]),
    ("Meta-expert", results_meta["expert"]),
]:
    returned = len(results)
    correct = sum(1 for r in results if r["page_url"] in GROUND_TRUTH)
    prec = correct / returned if returned > 0 else 0
    rec = correct / len(GROUND_TRUTH)
    print(f"{name:<20} {returned:>10} {correct:>10} {prec:>10.1%} {rec:>10.1%}")

Method                 Returned    Correct  Precision     Recall
--------------------------------------------------------------
Zero-shot @20                20          6      30.0%      21.4%
RAG-painting-KB @20          20         12      60.0%      42.9%
Meta-general                 13          7      53.8%      25.0%
Meta-basic                    1          1     100.0%       3.6%
Meta-expert                   3          3     100.0%      10.7%
